In [4]:
import pandas as pd
import json
from aksharamukha import transliterate
from graphviz import Digraph

In [3]:

pip install python-docx

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
   
def list_files_with_full_path(folder_path):
    try:
        files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
        return files
    except FileNotFoundError:
        print("Folder not found.")
        return []
    except PermissionError:
        print("Permission denied.")
        return []


folder_path = "DCS_rigveda_parsed"  # Change this to your folder path
files = list_files_with_full_path(folder_path)

print("Files in folder:")
for file in files:
    print(file)

import re

lines = files

pattern = r"ṚV, \d+, \d+"  # Regular expression to match ṚV, 1, X

extracted = [re.search(pattern, line).group() for line in lines if re.search(pattern, line)]
print(extracted)

Files in folder:
DCS_rigveda_parsed\Ṛgveda-0000-ṚV, 1, 1-9981.conllu_parsed
DCS_rigveda_parsed\Ṛgveda-0001-ṚV, 1, 2-9982.conllu_parsed
DCS_rigveda_parsed\Ṛgveda-0002-ṚV, 1, 3-9983.conllu_parsed
DCS_rigveda_parsed\Ṛgveda-0003-ṚV, 1, 4-9984.conllu_parsed
DCS_rigveda_parsed\Ṛgveda-0004-ṚV, 1, 5-9985.conllu_parsed
DCS_rigveda_parsed\Ṛgveda-0005-ṚV, 1, 6-9986.conllu_parsed
DCS_rigveda_parsed\Ṛgveda-0006-ṚV, 1, 7-9987.conllu_parsed
DCS_rigveda_parsed\Ṛgveda-0007-ṚV, 1, 8-9988.conllu_parsed
DCS_rigveda_parsed\Ṛgveda-0008-ṚV, 1, 9-9989.conllu_parsed
['ṚV, 1, 1', 'ṚV, 1, 2', 'ṚV, 1, 3', 'ṚV, 1, 4', 'ṚV, 1, 5', 'ṚV, 1, 6', 'ṚV, 1, 7', 'ṚV, 1, 8', 'ṚV, 1, 9']


In [6]:
def is_child_of(y, node, children_dict):
    return any(child == node for child, _ in children_dict.get(y, []))
def has_child_with_label(y, label, children_dict):
    return any(edge_label == label for _, edge_label in children_dict.get(y, []))
def is_child_with_label(y, node, label, children_dict):
    return any(child == node and edge_label == label for child, edge_label in children_dict.get(y, []))

In [7]:
from graphviz import Digraph
from collections import defaultdict
from IPython.display import Image, display

def transform_dependency_graph_rule_1(nodes, transformed_edges_set_1, output_prefix="dependency_graph", g = False):
    """
    Visualizes the original and transformed dependency graphs.

    RULE 5 in the Dependecy Paper

    The following rules have been applied:
    Rule 1a)
    if R1 is NOT one of the following -- kartaa, karma, lara.na, apaadaana, and sampradaana
    A <--- conjunct ---- B <----- R1 ----- C
    ====>    A <----- R1 ----- C  and   B <---- R1 ----- C

    Rule 1b)
    If R1 is one of the following -- kartaa, karma, lara.na, apaadaana, and sampradaana
    A <--- conjunct ---- B <----- R1 ----- C
    ====>    A <----- विशेषणम्  ----- B  and   B <---- R1 ----- C


    Args:
        nodes (dict): Mapping of node IDs to labels.
        transformed_edges_set_1 (list of tuples): List of (source, target, label) edges.
        output_prefix (str): Prefix for the output file names.
    """
    # ----------- ORIGINAL GRAPH -----------
    graph_original = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_original.node(node, label=f"{label}")
    
    # Add original edges
    for src, tgt, label in transformed_edges_set_1:
        graph_original.edge(src, tgt, label=label)

    # Render and display
    if g:
        original_path = f"{output_prefix}_original"
        graph_original.render(original_path, format="png", cleanup=True)
        display(Image(filename=original_path + ".png"))

    # ----------- TRANSFORMATION LOGIC -----------
    graph = defaultdict(list)  # child -> list of (parent, relation)
    
    for parent, child, relation in transformed_edges_set_1:
        graph[child].append((parent, relation))
    
    transformed_edges = []

    for child, parents in graph.items():
        for parent, relation in parents:
            if relation == 'Samuccita' and parent in graph:
                # If parent has a parent with any relation, add it as a grandparent link
                for grandparent, grand_rel in graph[parent]:
                    if grand_rel not in ["Kartā", "Karma", "larana", "apaadaana", "Sampradānam"]:
                        transformed_edges.append((grandparent, child, grand_rel))
                    else:
                        transformed_edges.append((parent, child, 'Viśeṣaṇam'))    
            else:
                transformed_edges.append((parent, child, relation))
    if g:
        print("Transformed Edges:")
        for edge in transformed_edges:
            print(edge)

    # ----------- TRANSFORMED GRAPH -----------
    graph_transformed = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_transformed.node(node, label=f"{label}")
    
    # Add transformed edges
    for src, tgt, label in transformed_edges:
        graph_transformed.edge(src, tgt, label=label)
    
    # Render and display
    
    if g:
        transformed_path = f"{output_prefix}_transformed"
        graph_transformed.render(transformed_path, format="png", cleanup=True)
        display(Image(filename=transformed_path + ".png"))
    return transformed_edges


In [8]:
from graphviz import Digraph
from collections import defaultdict
from IPython.display import Image, display

def transform_dependency_graph_rule_2(nodes, transformed_edges_set_1, output_prefix="dependency_graph", g = False):
    """
    Visualizes the original and transformed dependency graphs.

    RULE 2 in the Dependecy Paper
    
    The following rules have been applied:
    2.	For adjacent words:
    X <---- Conjunct ---- X 
    =====>  X <---- वीप्सा ----- X
    (Note both the words at the arrowhead and the arrow tail are the SAME)

    Args:
        nodes (dict): Mapping of node IDs to labels.
        transformed_edges_set_1 (list of tuples): List of (source, target, label) edges.
        output_prefix (str): Prefix for the output file names.
    """
    # ----------- ORIGINAL GRAPH -----------
    graph_original = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_original.node(node, label=f"{label}")
    
    # Add original edges
    for src, tgt, label in transformed_edges_set_1:
        graph_original.edge(src, tgt, label=label)

    # Render and display
    if g:
        original_path = f"{output_prefix}_original"
        graph_original.render(original_path, format="png", cleanup=True)
        display(Image(filename=original_path + ".png"))

    # ----------- TRANSFORMATION LOGIC -----------
    
    sorted_node_items = sorted(nodes.items())  # sorted by keys like '0_1', '0_2', etc.

    # Step 2: Compare adjacent words
    matching_ids = []
    for i in range(len(sorted_node_items) - 1):
        (id1, word1), (id2, word2) = sorted_node_items[i], sorted_node_items[i + 1]
        if word1 == word2:
            matching_ids.append((id1, id2))
    
    transformed_edges = []
    
    
    

    for edge in transformed_edges_set_1:
        src, tgt, label = edge
        if ((src, tgt) in matching_ids or (tgt, src) in matching_ids) and label == 'Samuccita':
            transformed_edges.append((src, tgt, 'vīpsā'))
        else:
            transformed_edges.append(edge)
        
        
    if g:
        print("Transformed Edges:")
        for edge in transformed_edges:
            print(edge)

    # ----------- TRANSFORMED GRAPH -----------
    graph_transformed = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_transformed.node(node, label=f"{label}")
    
    # Add transformed edges
    for src, tgt, label in transformed_edges:
        graph_transformed.edge(src, tgt, label=label)
    
    # Render and display
    if g:
        transformed_path = f"{output_prefix}_transformed"
        graph_transformed.render(transformed_path, format="png", cleanup=True)
        display(Image(filename=transformed_path + ".png"))
    return transformed_edges

In [9]:
from graphviz import Digraph
from collections import defaultdict
from IPython.display import Image, display

def transform_dependency_graph_rule_3(nodes, transformed_edges_set_1, output_prefix="dependency_graph", g = False):
    """
    Visualizes the original and transformed dependency graphs.

    RULE 4 in the Dependecy Paper
    
    The following rules have been applied:
    3.	A <----- विशेषणम्  ----- B <----- विशेषणम् ------ C
    =====> A <----- विशेषणम्  ----- C  and B <----- विशेषणम्  ----- C

    Args:
        nodes (dict): Mapping of node IDs to labels.
        transformed_edges_set_1 (list of tuples): List of (source, target, label) edges.
        output_prefix (str): Prefix for the output file names.
    """
    # ----------- ORIGINAL GRAPH -----------
    graph_original = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_original.node(node, label=f"{label}")
    
    # Add original edges
    for src, tgt, label in transformed_edges_set_1:
        graph_original.edge(src, tgt, label=label)

    # Render and display
    if g:
        original_path = f"{output_prefix}_original"
        graph_original.render(original_path, format="png", cleanup=True)
        display(Image(filename=original_path + ".png"))

    # ----------- TRANSFORMATION LOGIC -----------
    graph = defaultdict(list)  # child -> list of (parent, relation)
    
    for parent, child, relation in transformed_edges_set_1:
        graph[child].append((parent, relation))
    
    transformed_edges = []

    for child, parents in graph.items():
        for parent, relation in parents:
            if relation == 'Viśeṣaṇam' and parent in graph:
                # If parent has a parent with any relation, add it as a grandparent link
                for grandparent, grand_rel in graph[parent]:
                    if grand_rel == 'Viśeṣaṇam':
                        transformed_edges.append((grandparent, child, grand_rel))
                    else:
                        transformed_edges.append((parent, child, relation))
            else:
                transformed_edges.append((parent, child, relation))
    if g:
        print("Transformed Edges:")
        for edge in transformed_edges:
            print(edge)

    # ----------- TRANSFORMED GRAPH -----------
    graph_transformed = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_transformed.node(node, label=f"{label}")
    
    # Add transformed edges
    for src, tgt, label in transformed_edges:
        graph_transformed.edge(src, tgt, label=label)
    
    # Render and display
    if g:
        transformed_path = f"{output_prefix}_transformed"
        graph_transformed.render(transformed_path, format="png", cleanup=True)
        display(Image(filename=transformed_path + ".png"))
    return transformed_edges


In [10]:
from graphviz import Digraph
from collections import defaultdict
from IPython.display import Image, display

def transform_dependency_graph_rule_4(nodes, transformed_edges_set_1, output_prefix="dependency_graph", g = False):
    """
    Visualizes the original and transformed dependency graphs.

    RULE 3 in the Dependecy Paper
    
    The following rules have been applied:
    4.	For समिच्चित_द्योतकः, if parent <----- Conjuct----- grandparent
    Then add child to grandparent with link type समिच्चित_द्योतकः

    Args:
        nodes (dict): Mapping of node IDs to labels.
        transformed_edges_set_1 (list of tuples): List of (source, target, label) edges.
        output_prefix (str): Prefix for the output file names.
    """
    # ----------- ORIGINAL GRAPH -----------
    graph_original = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_original.node(node, label=f"{label}")
    
    # Add original edges
    for src, tgt, label in transformed_edges_set_1:
        graph_original.edge(src, tgt, label=label)

    # Render and display
    if g:
        original_path = f"{output_prefix}_original"
        graph_original.render(original_path, format="png", cleanup=True)
        display(Image(filename=original_path + ".png"))

    # ----------- TRANSFORMATION LOGIC -----------
    graph = defaultdict(list)  # child -> list of (parent, relation)
    
    for parent, child, relation in transformed_edges_set_1:
        graph[child].append((parent, relation))
    
    transformed_edges = []

    for child, parents in graph.items():
        for parent, relation in parents:
            if relation == 'Samiccita_dyotakaḥ' and parent in graph:
                # If parent has a parent with any relation, add it as a grandparent link
                for grandparent, grand_rel in graph[parent]:
                    if grand_rel == 'Samuccita':
                        transformed_edges.append((grandparent, child, relation))
                    else:
                        transformed_edges.append((parent, child, relation))
            else:
                transformed_edges.append((parent, child, relation))
    if g:
        print("Transformed Edges:")
        for edge in transformed_edges:
            print(edge)

    # ----------- TRANSFORMED GRAPH -----------
    graph_transformed = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_transformed.node(node, label=f"{label}")
    
    # Add transformed edges
    for src, tgt, label in transformed_edges:
        graph_transformed.edge(src, tgt, label=label)
    
    # Render and display
    if g:
        transformed_path = f"{output_prefix}_transformed"
        graph_transformed.render(transformed_path, format="png", cleanup=True)
        display(Image(filename=transformed_path + ".png"))
    return transformed_edges


In [11]:
from graphviz import Digraph
from collections import defaultdict
from IPython.display import Image, display

def transform_dependency_graph_rule_5(nodes, transformed_edges_set_1, output_prefix="dependency_graph", g = False):
    """
    Visualizes the original and transformed dependency graphs.

    RULE 1 in the Dependecy Paper
    
     A <--- समिच्चित_द्योतकः ---- B <--- conjunct ---- C <----- R1 ----- D 
    ====>    B <----- conjunct ----- D  and   C <---- R1 ----- D


    Args:
        nodes (dict): Mapping of node IDs to labels.
        transformed_edges_set_1 (list of tuples): List of (source, target, label) edges.
        output_prefix (str): Prefix for the output file names.
    """
    # ----------- ORIGINAL GRAPH -----------
    graph_original = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_original.node(node, label=f"{label}")
    
    # Add original edges
    for src, tgt, label in transformed_edges_set_1:
        graph_original.edge(src, tgt, label=label)

    # Render and display
    if g:
        original_path = f"{output_prefix}_original"
        graph_original.render(original_path, format="png", cleanup=True)
        display(Image(filename=original_path + ".png"))

    # ----------- TRANSFORMATION LOGIC -----------
    graph = defaultdict(list)  # child -> list of (parent, relation)
    
    for parent, child, relation in transformed_edges_set_1:
        graph[child].append((parent, relation))
    
    transformed_edges = []

    for child, parents in graph.items():
        for parent, relation in parents:
            if relation == 'Samuccita_dyotakaḥ' and parent in graph:
                # If parent has a parent with any relation, add it as a grandparent link
                for grandparent, grand_rel in graph[parent]:
                    if grand_rel == 'Samuccita' and grandparent in graph:
                        for great_grandparent, great_grand_rel in graph[grandparent]:
#                             transformed_edges.append((great_grandparent, parent, 'Samuccita'))
                            transformed_edges.append((parent, child, relation))
                            print(great_grandparent,"->",great_grand_rel,"->",grandparent,"->",grand_rel,"->",parent,relation,"->",child )
#                     if grand_rel not in ["Kartā", "Karma", "larana", "apaadaana", "Sampradānam"]:
#                         transformed_edges.append((grandparent, child, grand_rel))
#                     else:
#                         transformed_edges.append((parent, child, 'Viśeṣaṇam'))    
            else:
                transformed_edges.append((parent, child, relation))
    if g:
        print("Transformed Edges:")
        for edge in transformed_edges:
            print(edge)

    # ----------- TRANSFORMED GRAPH -----------
    graph_transformed = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_transformed.node(node, label=f"{label}")
    
    # Add transformed edges
    for src, tgt, label in transformed_edges:
        graph_transformed.edge(src, tgt, label=label)
    
    # Render and display
    
    if g:
        transformed_path = f"{output_prefix}_transformed"
        graph_transformed.render(transformed_path, format="png", cleanup=True)
        display(Image(filename=transformed_path + ".png"))
    return transformed_edges


In [12]:
from graphviz import Digraph
from collections import defaultdict
from IPython.display import Image, display

def transform_dependency_graph_rule_5_a(nodes, transformed_edges_set_1, output_prefix="dependency_graph", g = False):
    """
    Visualizes the original and transformed dependency graphs.

    The following rules have been applied:
    Rule 1a)
    if R1 is NOT one of the following -- kartaa, karma, lara.na, apaadaana, and sampradaana
    A <--- conjunct ---- B <----- R1 ----- C
    ====>    A <----- R1 ----- C  and   B <---- R1 ----- C

    Rule 1b)
    If R1 is one of the following -- kartaa, karma, lara.na, apaadaana, and sampradaana
    A <--- conjunct ---- B <----- R1 ----- C
    ====>    A <----- विशेषणम्  ----- B  and   B <---- R1 ----- C


    Args:
        nodes (dict): Mapping of node IDs to labels.
        transformed_edges_set_1 (list of tuples): List of (source, target, label) edges.
        output_prefix (str): Prefix for the output file names.
    """
    # ----------- ORIGINAL GRAPH -----------
    graph_original = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_original.node(node, label=f"{label}")
    
    # Add original edges
    for src, tgt, label in transformed_edges_set_1:
        graph_original.edge(src, tgt, label=label)

    # Render and display
    if g:
        original_path = f"{output_prefix}_original"
        graph_original.render(original_path, format="png", cleanup=True)
        display(Image(filename=original_path + ".png"))

    # ----------- TRANSFORMATION LOGIC -----------
    graph = defaultdict(list)  # child -> list of (parent, relation)
    
    for parent, child, relation in transformed_edges_set_1:
        graph[child].append((parent, relation))
    
    transformed_edges = []

    for child, parents in graph.items():
        for parent, relation in parents:
            if relation == 'Samuccita' and parent in graph:
                # If parent has a parent with any relation, add it as a grandparent link
                for grandparent, grand_rel in graph[parent]:
                    transformed_edges.append((parent, child, 'Samuccita'))    
            else:
                transformed_edges.append((parent, child, relation))
            
            if relation == 'Samuccita_dyotakaḥ' and parent in graph:
                # If parent has a parent with any relation, add it as a grandparent link
                for grandparent, grand_rel in graph[parent]:
                    if grand_rel == 'Samuccita' and grandparent in graph:
                        for great_grandparent, great_grand_rel in graph[grandparent]:
                            transformed_edges.append((great_grandparent, parent, 'Samuccita'))
                            transformed_edges.append((parent, child, relation))
                            print(great_grandparent,"->",great_grand_rel,"->",grandparent,"->",grand_rel,"->",parent,relation,"->",child )
#                     if grand_rel not in ["Kartā", "Karma", "larana", "apaadaana", "Sampradānam"]:
#                         transformed_edges.append((grandparent, child, grand_rel))
#                     else:
#                         transformed_edges.append((parent, child, 'Viśeṣaṇam'))    
            else:
                transformed_edges.append((parent, child, relation))
    if g:
        print("Transformed Edges:")
        for edge in transformed_edges:
            print(edge)

    # ----------- TRANSFORMED GRAPH -----------
    graph_transformed = Digraph(format='png', engine='dot')
    
    # Add nodes
    for node, label in nodes.items():
        graph_transformed.node(node, label=f"{label}")
    
    # Add transformed edges
    for src, tgt, label in transformed_edges:
        graph_transformed.edge(src, tgt, label=label)
    
    # Render and display
    
    if g:
        transformed_path = f"{output_prefix}_transformed"
        graph_transformed.render(transformed_path, format="png", cleanup=True)
        display(Image(filename=transformed_path + ".png"))
    return transformed_edges


In [13]:
pip install docx

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [14]:
from docx import Document
from docx.shared import Inches
from docx.shared import Pt
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.enum.text import WD_BREAK
from docx.oxml.ns import nsdecls
from docx.shared import Pt, RGBColor

def set_paragraph_spacing(cell, space_before=5, space_after=5):
    for paragraph in cell.paragraphs:
        paragraph_format = paragraph.paragraph_format
        paragraph_format.space_before = Pt(space_before)
        paragraph_format.space_after = Pt(space_after)

# Function to add table borders
def set_borders(table):
    tbl = table._element  # Access table element
    ns = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}

    # Locate <w:tblPr> if it exists
    tbl_pr = tbl.find('.//w:tblPr', ns)  

    # If <w:tblPr> doesn't exist, create it
    if tbl_pr is None:
        tbl_pr = OxmlElement('w:tblPr')
        tbl.insert(0, tbl_pr)

    tbl_borders = OxmlElement('w:tblBorders')

    for border_name in ['top', 'left', 'bottom', 'right', 'insideH', 'insideV']:
        border = OxmlElement(f'w:{border_name}')
        border.set(qn('w:val'), 'single')     # Correct use of namespaces
        border.set(qn('w:sz'), '6')           # Thicker borders
        border.set(qn('w:space'), '0')        # Space inside borders
        border.set(qn('w:color'), '000000')   # Black color
        tbl_borders.append(border)

    tbl_pr.append(tbl_borders)

# Function to apply header style
def format_header(cell):
    cell.text = cell.text
    paragraph = cell.paragraphs[0]
    run = paragraph.runs[0]
    run.font.bold = True
    run.font.size = Pt(12)
    cell.paragraphs[0].alignment = 1  # Center alignment

    # Adding background color
    tc_pr = cell._element.find(qn("w:tcPr"))  # Correct namespace usage
    if tc_pr is None:
        tc_pr = OxmlElement('w:tcPr')  # Create <w:tcPr> if missing
        cell._element.insert(0, tc_pr)

    # Add shading
    shading_elm = OxmlElement('w:shd')
    shading_elm.set(qn('w:val'), 'clear')       # Transparent background
    shading_elm.set(qn('w:color'), 'auto')       # Auto text color
    shading_elm.set(qn('w:fill'), '004875')      # Dark blue background
    tc_pr.append(shading_elm)

    # Text color
    for run in cell.paragraphs[0].runs:
        run.font.color.rgb = RGBColor(255, 255, 255)  # White text

        
# Function to set column widths
def set_column_widths(table, widths):
    table.autofit = False
    for col_idx, width in enumerate(widths):
        for row in table.rows:
            cell = row.cells[col_idx]

            # Create or find cell properties <w:tcPr>
            tc_pr = cell._element.find(qn("w:tcPr"))
            if not tc_pr:
                tc_pr = OxmlElement('w:tcPr')
                cell._element.insert(0, tc_pr)

            # Add width specification
            tc_width = OxmlElement('w:tcW')
            tc_width.set(qn('w:w'), str(int(width * 1440)))  # Correct namespace
            tc_width.set(qn('w:type'), 'dxa')                # Fixed width
            tc_pr.append(tc_width)        
        

def apply_font_style(paragraph, font_name='Times New Roman', size=12, bold=False, italic=False):
    run = paragraph.add_run()
    run.font.name = font_name
    run.font.size = Pt(size)
    run.bold = bold
    run.italic = italic
    paragraph.alignment = WD_PARAGRAPH_ALIGNMENT.LEFT  # Alignment options: LEFT, CENTER, RIGHT, JUSTIFY

In [ ]:
# import re
# from collections import defaultdict
# from collections import defaultdict
# from PIL import Image as PILImage

# def is_child_of(y, node, children_dict):
#     return any(child == node for child, _ in children_dict.get(y, []))
# def has_child_with_label(y, label, children_dict):
#     return any(edge_label == label for _, edge_label in children_dict.get(y, []))
# def is_child_with_label(y, node, label, children_dict):
#     return any(child == node and edge_label == label for child, edge_label in children_dict.get(y, []))

# label_to_sanskrit_form = {
#     "Object": "कर्म",
#     "Nominal modifier": "विशेषणम्",
#     "Conjunct": "समुच्चित",
#     "Coordinating conjunction": "समुच्चित_द्योतकःः",
#     "Vocative": "सम्बोध्यः",
#     "Indirect object": "सम्प्रदानम्",
#     "Determiner": "विशेषणम्",
#     "Clausal modifier of noun (adnominal clause)": "विशेषणम्",
#     "Adjectival modifier": "विशेषणम्",
#     "nmod:appos_Unknown_label": "विशेषणम्",
#     "Adverbial modifier": "क्रियाविशेषणम्",
#     "Adverbial clause modifier": "क्रियाविशेषणम्",
#     "obj:instr_Unknown_label":"करणम्",
#     "Numeric modifier":"विशेषणम्",
#     "Discourse/Dislocated element":"सम्बन्धः",
#     "Case marking":"कर्मप्रवचनीयः",
    
    
# }

# transform_1_labels = list(label_to_sanskrit_form.keys())

# transform_2_labels = ['Oblique nominal', "Nominal modifier", 'Nominal subject', "Coordinating conjunction"]

# label_to_sanskrit_form_iast = {
#     key: transliterate.process('Devanagari', 'IAST', value).capitalize()
#     for key, value in label_to_sanskrit_form.items()
# }
# label_to_full_form = {
#     "root": "Root",
#     "mark": "Marker",
#     "flat": "Flat expression",
#     "nummod": "Numeric modifier",
#     "vocative": "Vocative",
#     "obj": "Object",
#     "advmod": "Adverbial modifier",
#     "cc": "Coordinating conjunction",
#     "nsubj": "Nominal subject",
#     "fixed": "Fixed multiword expression",
#     "obl": "Oblique nominal",
#     "discourse/dislocated": "Discourse/Dislocated element",
#     "det": "Determiner",
#     "advcl": "Adverbial clause modifier",
#     "amod": "Adjectival modifier",
#     "aux": "Auxiliary",
#     "nmod": "Nominal modifier",
#     "iobj": "Indirect object",
#     "cop": "Copula",
#     "case": "Case marking",
#     "ccomp": "Clausal complement",
#     "compound": "Compound",
#     "conj": "Conjunct",
#     "acl": "Clausal modifier of noun (adnominal clause)",
#     "xcomp": "Open clausal complement",
#     "appos": "Appositional modifier",
#     "csubj": "Clausal subject",
#     "orphan": "Orphan",
#     "parataxis": "Parataxis",
# }

# def extract_and_group_columns(data):
#     grouped_sentences = defaultdict(list)
    
#     for line in data.split('\n'):
#         if line.startswith("# text ="):
#             current_text = line.split("= ", 1)[1]
#         elif line.startswith("# sent_id ="):
# #             print(line)
#             current_sent_id = line.split("= ", 1)[1]
# #             print(current_sent_id)
#             if "_" not in current_sent_id:
#                 split_sent_id = [current_sent_id,"0"]
#             else:   
#                 split_sent_id = current_sent_id.split("_")
# #             print(split_sent_id)
#             group_key = split_sent_id[0]  # Group by sent_id[0]
#         elif line and not line.startswith("#"):
#             columns = line.split('\t')
#             if len(columns) > 1:
# #                 print(columns)
#                 column_id = columns[0]
#                 parent_id = columns[6]
#                 updated_column_id = f"{split_sent_id[1]}_{column_id}"  # Update column ID with sent_id[2]
#                 columns[0] = updated_column_id
#                 updated_parent_id = f"{split_sent_id[1]}_{parent_id}"  # Update column ID with sent_id[2]
#                 columns[6] = updated_parent_id
#                 grouped_sentences[group_key].append((current_text, columns))
    
#     return grouped_sentences

# # Example usage
# # data = """(Your provided data here)"""

# # file_path = 'RV_1_4.conllu'
# # file_path = "Ṛgveda-0003-ṚV, 1, 4-9984.conllu_parsed"
# # file_path = files[3]

# # files = [files[3]]
# j=0

# arrow_line_color_green = "#74c476"
# font_color_magenta = "#6d000b"
# font_color_purple = "#63006d"
# font_color_green = "#006d2c"

# theme_m = [arrow_line_color_green,font_color_magenta]
# theme_v = [arrow_line_color_green,font_color_purple]
# theme_g = [arrow_line_color_green,font_color_green]

# num = 0
# doc = Document()

# for file_path in files:
#     heading = doc.add_heading(f"ConLLU File: {extracted[j]}", level=1)
#     apply_font_style(heading, font_name='Calibri', size=14, bold=True)
#     print("=" * 50)
#     print("\033[1mConLLU File:\033[0m",extracted[j])
#     print("=" * 50)
#     j=j+1
#     with open(file_path, 'r', encoding='utf-8') as file:
#         data = file.read()


#     grouped_data = extract_and_group_columns(data)
#     text_combined_list =[]
#     group_id_list =[]
#     # Print grouped data
#     for group, sentences in grouped_data.items():
#     #     print(f"Group: {group}")
#         text_list = []
#         for text, columns in sentences:
#             text_list.append(text)
#     #         print(f"Text: {text}")
#     #         print(f"Columns: {columns}")
#         text_unique = list(dict.fromkeys(text_list))
#         text_combined_list.append(" ".join(text_unique))
#         group_id_list.append(group)
#     #     print("=" * 50)


#     list_dcs_json =[]
#     for group, sentences in grouped_data.items():
#     #     print(f"Group: {group}")
#         col = []
#         for text, columns in sentences:
#             word = columns[1] if columns[1]!="_" else columns[2]
#             col.append((columns[0],word,columns[7],columns[6], columns[5])) 
#     #         print(col)
#     #         print(f"Text: {text}")
#     #         print(f"Columns: {columns}")
#         df = pd.DataFrame(col, columns=['Index', 'Word', 'Link_Type', 'Parent', 'Grammar'])
#         json_temp = df.to_json(orient='records', force_ascii=False)
#         formatted_json = json.dumps(json.loads(json_temp), indent=4, ensure_ascii=False)
#     #     print(formatted_json)
#         list_dcs_json.append(json_temp)

#     #     print("=" * 50)    


#     i = 0
#     for x in list_dcs_json:

#             data = json.loads(x)

#             graph_1 = Digraph(format='png', engine='dot')
#             graph_2 = Digraph(format='png', engine='dot')

#             # Extract nodes and edges
#             nodes = {str(item["Index"]): item["Word"] for item in data if item["Link_Type"] not in ["_"]}
#             edges = [(str(item["Parent"]), str(item["Index"]), label_to_sanskrit_form.get(item["Link_Type"], item["Link_Type"])) for item in data if item["Link_Type"] not in ["root", "_"] ]

#             # Add nodes
#             for node, label in nodes.items():
#                 graph_1.node(node, label=f"{label}",color="#3182bd",shape="box", style="rounded,filled", fillcolor="#f7fbff", fontcolor ="#08519c")
#                 graph_2.node(node, label=f"{label}",color="#00441b",shape="box", style="rounded,filled", fillcolor="#f7fcf5", fontcolor ="#006d2c")

#             transformed_edges_set_1 = []
#             # Add edges with labels
            
#             for parent_temp, child_temp, label_temp in edges:
#                 children_dict[parent_temp].append((child_temp, label_temp))
            
#             for src, tgt, label in edges:
#                 label_temp=label_to_full_form.get(label, label+"_Unknown_label")
#                 graph_1.edge(src, tgt, label=label_to_full_form.get(label, label+"_Unknown_label"), color="#6baed6", fontcolor ="#08519c")
#                 if label_temp in transform_2_labels:
#                     for word_info in data:
#                         if word_info["Index"] == src:
#                             parent_grammar = word_info["Grammar"]
# #                             print("Parent is ", word_info["Word"])
#                         if word_info["Index"] == tgt:
#                             child_grammar = word_info["Grammar"]
#                             child_word = word_info["Word"]
# #                             print("Child is ", word_info["Word"])
                    
#                     if (label_temp == "Nominal modifier" and "Case=Gen" in child_grammar):
#                         value = "षष्ठीसम्बन्धः"
#                         color_selected, font_color_selected = theme_v
# #                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
#                     elif (label_temp == "Nominal modifier"): # Addingthis here because it is added in the list of second labels to be transformed
#                         value = label_to_sanskrit_form_iast.get(label_temp, label_temp)
#                         color_selected, font_color_selected = theme_v
# #                         graph_2.edge(src, tgt, label=label_to_sanskrit_form_iast.get(label_temp, label_temp), color="#74c476", fontcolor ="#6d000b")
                    
#                     elif (label_temp == 'Oblique nominal' and "Case=Loc" in child_grammar):
#                         value = "अधिकरणम्"
#                         color_selected, font_color_selected = theme_v
# #                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
#                     elif (label_temp == 'Oblique nominal' and "Case=Dat" in child_grammar):
#                         value = "सम्प्रदानम्"
#                         color_selected, font_color_selected = theme_v
                    
#                     elif (label_temp == 'Oblique nominal' and "Case=Abl" in child_grammar):
#                         value = "अपादानम्"
#                         color_selected, font_color_selected = theme_v
                    
#                     elif (label_temp == 'Oblique nominal' and "Case=Acc" in child_grammar and has_child_with_label(tgt, 'cas', children_dict)):
#                         value = "कर्मप्रवचनीय_अन्वितः"
#                         print("कर्मप्रवचनीय_अन्वितः")
#                         color_selected, font_color_selected = theme_v
                    
#                     elif (label_temp == 'Oblique nominal' and "Case=Acc" in child_grammar):
#                         value = "कर्म"
#                         color_selected, font_color_selected = theme_v
                        
                    
# #                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
#                     elif (label_temp == 'Oblique nominal' and "Case=Ins" in child_grammar and ("Ṇyat" in parent_grammar or "Gdv" in parent_grammar)):
#                         value = "कर्ता" 
#                         color_selected, font_color_selected = theme_v
# #                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
#                     elif (label_temp == 'Oblique nominal') and ("Case=Ins" in child_grammar):
#                         value = "करणम्"
#                         color_selected, font_color_selected = theme_v
# #                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
#                     elif (label_temp == "Coordinating conjunction") and (child_word  == "vā"):
#                         value = "अन्यतर_द्योतकः"
#                         color_selected, font_color_selected = theme_v                    
                    
#                     elif (label_temp == 'Nominal subject'):
#                         if "Ṇyat" in parent_grammar or "Gdv" in parent_grammar:
#                             value = "कर्म" 
#                             color_selected, font_color_selected = theme_v
# #                             graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
#                         else:
#                             value = "कर्ता"
#                             color_selected, font_color_selected = theme_v
# #                             graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
#                     else:
#                         value = label_to_sanskrit_form_iast.get(label_temp, label_temp)
#                         color_selected, font_color_selected = theme_g
# #                         graph_2.edge(src, tgt, label=label_to_sanskrit_form_iast.get(label_temp, label_temp), color="#74c476", fontcolor ="#006d2c")
                    
# #                     transformed_edges_set_1.append([src,tgt,transformed_label_set1])            
#                 elif label_temp in transform_1_labels:
#                     value = label_to_sanskrit_form_iast.get(label_temp, label_temp)
#                     color_selected, font_color_selected = theme_m
# #                     graph_2.edge(src, tgt, label=label_to_sanskrit_form_iast.get(label_temp, label_temp), color="#74c476", fontcolor ="#6d000b")
#                 else:
#                     value = label_to_sanskrit_form_iast.get(label_temp, label_temp)
#                     color_selected, font_color_selected = theme_g
# #                     graph_2.edge(src, tgt, label=label_to_sanskrit_form_iast.get(label_temp, label_temp), color="#74c476", fontcolor ="#006d2c")
#                 value_transliterated = transliterate.process('Devanagari', 'IAST', value).capitalize()
#                 transformed_edges_set_1.append((src, tgt, value_transliterated))
#                 graph_2.edge(src, tgt, label=value_transliterated, color=color_selected, fontcolor = font_color_selected)
            
            
            
            
#             # Render and save the graph
#             graph_1_filepath = "dependency_graph_1"
#             graph_2_filepath = "dependency_graph_2"
#             graph_1.render(graph_1_filepath, format="png", cleanup=True)
#             graph_2.render(graph_2_filepath, format="png", cleanup=True)
            
            

#             # Display the rendered graph
#             print("\033[1mMantra Text:\033[0m",text_combined_list[i])
#             print("\033[1mSentence ID:\033[0m", group_id_list[i])
# #             i = i+1
#             from IPython.display import Image, display
#             print("\033[1mGraph Style:\033[0m", "DCS Original")
#             display(Image(filename=graph_1_filepath + ".png")) 
#             print("\033[1mGraph Style:\033[0m", "Modified from DCS")
#             display(Image(filename=graph_2_filepath + ".png"))
# #             print("First Transformeation Edges:")
# #             for edge in transformed_edges_set_1:
# #                 print(edge)
            
#             """The Rule order is Rule 2, Rule 4, Rule 3, and finally Rule1
#             The following rules have been applied:
#             Rule 1a)
#             if R1 is NOT one of the following -- kartaa, karma, lara.na, apaadaana, and sampradaana
#             A <--- conjunct ---- B <----- R1 ----- C
#             ====>    A <----- R1 ----- C  and   B <---- R1 ----- C

#             Rule 1b)
#             If R1 is one of the following -- kartaa, karma, lara.na, apaadaana, and sampradaana
#             A <--- conjunct ---- B <----- R1 ----- C
#             ====>    A <----- विशेषणम्  ----- B  and   B <---- R1 ----- C

#             2.	For adjacent words:
#             X <---- Conjunct ---- X 
#             =====>  X <---- वीप्सा ----- X
#             (Note both the words at the arrowhead and the arrow tail are the SAME)

#             3.	A <----- विशेषणम्  ----- B <----- विशेषणम् ------ C
#             =====> A <----- विशेषणम्  ----- C  and B <----- विशेषणम्  ----- C

#             4.	For समिच्चित_द्योतकः, if parent <----- Conjuct----- grandparent
#             Then add child to grandparent with link type समिच्चित_द्योतकः
#             """
        
#             transformed_edges_set_2 = transform_dependency_graph_rule_2(nodes, transformed_edges_set_1, output_prefix="dependency_graph")
#             transformed_edges_set_3 = transform_dependency_graph_rule_4(nodes, transformed_edges_set_2, output_prefix="dependency_graph")
#             transformed_edges_set_4 = transform_dependency_graph_rule_3(nodes, transformed_edges_set_3, output_prefix="dependency_graph")
#             transformed_edges_set_5 = transform_dependency_graph_rule_1(nodes, transformed_edges_set_4, output_prefix="dependency_graph", g = False)
#             if (False):
#                 print("Transformation Edges 1:")
#                 for edge in transformed_edges_set_1:
#                     print(edge)
#                 print("Transformation Edges 2:")
#                 for edge in transformed_edges_set_2:
#                     print(edge)
#                 print("Transformation Edges 3:")
#                 for edge in transformed_edges_set_3:
#                     print(edge)
#                 print("Transformation Edges: 4")
#                 for edge in transformed_edges_set_4:
#                     print(edge)
#                 print("Transformation Edges: 5 ")
#                 for edge in transformed_edges_set_5:
#                     print(edge)
#             graph_reduction = Digraph(format='png', engine='dot')
    
#             # Add nodes
#             for node, label in nodes.items():
#                 graph_reduction.node(node, label=f"{label}",color="#cf3759",shape="box", style="rounded,filled", fillcolor="#FFEEEB", fontcolor ="#93003a")

#             # Add original edges
#             for src, tgt, label in transformed_edges_set_5:
#                 graph_reduction.edge(src, tgt, label=label, color="#f4777f", fontcolor ="#93003a")

#             # Render and display
           
#             reduction_path = f"dependency_graph_3"
#             graph_reduction.render(reduction_path, format="png", cleanup=True)
#             print("\033[1mGraph Style:\033[0m", "Reduced from Modified Graph")
#             display(Image(filename=reduction_path + ".png"))
            
            
#             heading = doc.add_heading(f"Sentence ID: {group_id_list[i]}", level=2)
#             apply_font_style(heading, font_name='Calibri', size=14, bold=True)

#             caption_v = doc.add_paragraph()
#             run = caption_v.add_run(f"Mantra Text:\n{text_combined_list[i]}")
# #             run.bold = True
#             run.font.size = Pt(12)

#             # Add dependency parse
# #             parse_paragraph = doc.add_paragraph(f"Dependency Parsing from ByT5:\n{parse[num]}")
# #             apply_font_style(parse_paragraph, font_name='Calibri', size=12)

#             # Adding images
#             image_width_limit = 6
#             image_path_1 = graph_1_filepath + ".png"
        
#             with PILImage.open(image_path_1) as img:
#                 width_px, height_px = img.size
#                 dpi = img.info.get('dpi', (96, 96))[0]  # Default to 96 DPI if not specified
#                 width_in = width_px / dpi
            
# #             caption_v = doc.add_paragraph()
# #             run = caption_v.add_run(f"Width pixel:{width_px} \n Width inch:{width_in}")
#             # Add the image with resizing only if width > 4 inches
#             if width_in > image_width_limit:
#                 doc.add_picture(image_path_1, width=Inches(image_width_limit))
#             else:
#                 doc.add_picture(image_path_1, width=Inches(width_in))
# #             doc.add_picture(image_path_1, width=Inches(4))
    
#             caption_1 = doc.add_paragraph()
#             run = caption_1.add_run(f"Figure {num * 3 + 1}: DCS Original Dependency Graph for Sentence ID {group_id_list[i]}")
#             run.italic = True
#             run.font.size = Pt(10)
#             caption_1.paragraph_format.space_after = Pt(20)
            
#             image_path_2 = graph_2_filepath + ".png"
# #             doc.add_picture(image_path_2, width=Inches(4))
#             with PILImage.open(image_path_2) as img:
#                 width_px, height_px = img.size
#                 dpi = img.info.get('dpi', (96, 96))[0]  # Default to 96 DPI if not specified
#                 width_in = width_px / dpi
# #             caption_v = doc.add_paragraph()
# #             run = caption_v.add_run(f"Width pixel:{width_px} \n Width inch:{width_in}")
#             # Add the image with resizing only if width > 4 inches
#             if width_in > image_width_limit:
#                 doc.add_picture(image_path_2, width=Inches(image_width_limit))
#             else:
#                 doc.add_picture(image_path_2, width=Inches(width_in))
   
#             caption_2 = doc.add_paragraph()
#             run = caption_2.add_run(f"Figure {num * 3 + 2}: Sanskrit Tags Dependency Graph for Sentence ID {group_id_list[i]}")
#             run.italic = True
#             run.font.size = Pt(10)
#             caption_2.paragraph_format.space_after = Pt(20)

#             image_path_3 = reduction_path + ".png"
# #             doc.add_picture(image_path_3, width=Inches(4))
#             with PILImage.open(image_path_3) as img:
#                 width_px, height_px = img.size
#                 dpi = img.info.get('dpi', (96, 96))[0]  # Default to 96 DPI if not specified
#                 width_in = width_px / dpi
# #             caption_v = doc.add_paragraph()
# #             run = caption_v.add_run(f"Width pixel:{width_px} \n Width inch:{width_in}")
#             # Add the image with resizing only if width > 4 inches
#             if width_in > image_width_limit:
#                 doc.add_picture(image_path_3, width=Inches(image_width_limit))
#             else:
#                 doc.add_picture(image_path_3, width=Inches(width_in))
            
            
#             caption_3 = doc.add_paragraph()
#             run = caption_3.add_run(f"Figure {num * 3 + 3}: Reduced Dependency Graph for Sentence ID {group_id_list[i]}")
#             run.italic = True
#             run.font.size = Pt(10)
#             run.add_break(WD_BREAK.PAGE)
            
#             i = i+1
#             num = num + 3
            
# #             break

# doc.save("Reduced_RigVeda_Dependency_Graph_DCS_Ver_x4.docx")
# print("Document successfully created!")

In [16]:
import re
from collections import defaultdict
from collections import defaultdict
from PIL import Image as PILImage

def is_child_of(y, node, children_dict):
    return any(child == node for child, _ in children_dict.get(y, []))
def has_child_with_label(y, label, children_dict):
    return any(edge_label == label for _, edge_label in children_dict.get(y, []))
def is_child_with_label(y, node, label, children_dict):
    return any(child == node and edge_label == label for child, edge_label in children_dict.get(y, []))

label_to_sanskrit_form = {
    "Object": "कर्म",
    "Nominal modifier": "विशेषणम्",
    "Conjunct": "समुच्चित",
    "Coordinating conjunction": "समुच्चित_द्योतकः",
    "Vocative": "सम्बोध्यः",
    "Indirect object": "सम्प्रदानम्",
    "Determiner": "विशेषणम्",
    "Clausal modifier of noun (adnominal clause)": "विशेषणम्",
    "Adjectival modifier": "विशेषणम्",
    "nmod:appos_Unknown_label": "विशेषणम्",
    "Adverbial modifier": "क्रियाविशेषणम्",
    "Adverbial clause modifier": "क्रियाविशेषणम्",
    "obj:instr_Unknown_label":"करणम्",
    "Numeric modifier":"विशेषणम्",
    "Discourse/Dislocated element":"सम्बन्धः",
    "Case marking":"कर्मप्रवचनीयः",
    
    
}

transform_1_labels = list(label_to_sanskrit_form.keys())

transform_2_labels = ['Oblique nominal', "Nominal modifier", 'Nominal subject', "Coordinating conjunction"]

label_to_sanskrit_form_iast = {
    key: transliterate.process('Devanagari', 'IAST', value).capitalize()
    for key, value in label_to_sanskrit_form.items()
}
label_to_full_form = {
    "root": "Root",
    "mark": "Marker",
    "flat": "Flat expression",
    "nummod": "Numeric modifier",
    "vocative": "Vocative",
    "obj": "Object",
    "advmod": "Adverbial modifier",
    "cc": "Coordinating conjunction",
    "nsubj": "Nominal subject",
    "fixed": "Fixed multiword expression",
    "obl": "Oblique nominal",
    "discourse/dislocated": "Discourse/Dislocated element",
    "det": "Determiner",
    "advcl": "Adverbial clause modifier",
    "amod": "Adjectival modifier",
    "aux": "Auxiliary",
    "nmod": "Nominal modifier",
    "iobj": "Indirect object",
    "cop": "Copula",
    "case": "Case marking",
    "ccomp": "Clausal complement",
    "compound": "Compound",
    "conj": "Conjunct",
    "acl": "Clausal modifier of noun (adnominal clause)",
    "xcomp": "Open clausal complement",
    "appos": "Appositional modifier",
    "csubj": "Clausal subject",
    "orphan": "Orphan",
    "parataxis": "Parataxis",
}

def extract_and_group_columns(data):
    grouped_sentences = defaultdict(list)
    
    for line in data.split('\n'):
        if line.startswith("# text ="):
            current_text = line.split("= ", 1)[1]
        elif line.startswith("# sent_id ="):
#             print(line)
            current_sent_id = line.split("= ", 1)[1]
#             print(current_sent_id)
            if "_" not in current_sent_id:
                split_sent_id = [current_sent_id,"0"]
            else:   
                split_sent_id = current_sent_id.split("_")
#             print(split_sent_id)
            group_key = split_sent_id[0]  # Group by sent_id[0]
        elif line and not line.startswith("#"):
            columns = line.split('\t')
            if len(columns) > 1:
#                 print(columns)
                column_id = columns[0]
                parent_id = columns[6]
                updated_column_id = f"{split_sent_id[1]}_{column_id}"  # Update column ID with sent_id[2]
                columns[0] = updated_column_id
                updated_parent_id = f"{split_sent_id[1]}_{parent_id}"  # Update column ID with sent_id[2]
                columns[6] = updated_parent_id
                grouped_sentences[group_key].append((current_text, columns))
    
    return grouped_sentences

# Example usage
# data = """(Your provided data here)"""

# file_path = 'RV_1_4.conllu'
# file_path = "Ṛgveda-0003-ṚV, 1, 4-9984.conllu_parsed"
# file_path = files[3]

# files = [files[3]]
j=0

arrow_line_color_green = "#74c476"
font_color_magenta = "#6d000b"
font_color_purple = "#63006d"
font_color_green = "#006d2c"

theme_m = [arrow_line_color_green,font_color_magenta]
theme_v = [arrow_line_color_green,font_color_purple]
theme_g = [arrow_line_color_green,font_color_green]

num = 0
doc = Document()

for file_path in files:
    heading = doc.add_heading(f"ConLLU File: {extracted[j]}", level=1)
    apply_font_style(heading, font_name='Calibri', size=14, bold=True)
    print("=" * 50)
    print("\033[1mConLLU File:\033[0m",extracted[j])
    print("=" * 50)
    j=j+1
    with open(file_path, 'r', encoding='utf-8') as file:
        data = file.read()


    grouped_data = extract_and_group_columns(data)
    text_combined_list =[]
    group_id_list =[]
    # Print grouped data
    for group, sentences in grouped_data.items():
    #     print(f"Group: {group}")
        text_list = []
        for text, columns in sentences:
            text_list.append(text)
    #         print(f"Text: {text}")
    #         print(f"Columns: {columns}")
        text_unique = list(dict.fromkeys(text_list))
        text_combined_list.append(" ".join(text_unique))
        group_id_list.append(group)
    #     print("=" * 50)


    list_dcs_json =[]
    for group, sentences in grouped_data.items():
    #     print(f"Group: {group}")
        col = []
        for text, columns in sentences:
            word = columns[1] if columns[1]!="_" else columns[2]
            col.append((columns[0],word,columns[7],columns[6], columns[5])) 
    #         print(col)
    #         print(f"Text: {text}")
    #         print(f"Columns: {columns}")
        df = pd.DataFrame(col, columns=['Index', 'Word', 'Link_Type', 'Parent', 'Grammar'])
        json_temp = df.to_json(orient='records', force_ascii=False)
        formatted_json = json.dumps(json.loads(json_temp), indent=4, ensure_ascii=False)
    #     print(formatted_json)
        list_dcs_json.append(json_temp)

    #     print("=" * 50)    


    i = 0
    for x in list_dcs_json:

            data = json.loads(x)

            graph_1 = Digraph(format='png', engine='dot')
            graph_2 = Digraph(format='png', engine='dot')

            # Extract nodes and edges
            nodes = {str(item["Index"]): item["Word"] for item in data if item["Link_Type"] not in ["_"]}
            edges = [(str(item["Parent"]), str(item["Index"]), label_to_sanskrit_form.get(item["Link_Type"], item["Link_Type"])) for item in data if item["Link_Type"] not in ["root", "_"] ]
            print(edges)
            # Add nodes
            for node, label in nodes.items():
                graph_1.node(node, label=f"{label}",color="#3182bd",shape="box", style="rounded,filled", fillcolor="#f7fbff", fontcolor ="#08519c")
                graph_2.node(node, label=f"{label}",color="#00441b",shape="box", style="rounded,filled", fillcolor="#f7fcf5", fontcolor ="#006d2c")

            transformed_edges_set_1 = []
            # Add edges with labels
            children_dict = defaultdict(list)
            for parent_temp, child_temp, label_temp in edges:
                children_dict[parent_temp].append((child_temp, label_temp))
            
            for src, tgt, label in edges:
                label_temp=label_to_full_form.get(label, label+"_Unknown_label")
                graph_1.edge(src, tgt, label=label_to_full_form.get(label, label+"_Unknown_label"), color="#6baed6", fontcolor ="#08519c")
                if label_temp in transform_2_labels:
                    for word_info in data:
                        if word_info["Index"] == src:
                            parent_grammar = word_info["Grammar"]
                            parent_word1 = word_info["Word"]
#                             print("Parent is ", word_info["Word"])
                        if word_info["Index"] == tgt:
                            child_grammar = word_info["Grammar"]
                            child_word = word_info["Word"]
#                             print("Child is ", word_info["Word"])
                    
                    if (label_temp == "Nominal modifier" and "Case=Gen" in child_grammar):
                        value = "षष्ठीसम्बन्धः"
                        color_selected, font_color_selected = theme_v
#                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
                    elif (label_temp == "Nominal modifier"): # Addingthis here because it is added in the list of second labels to be transformed
                        value = label_to_sanskrit_form_iast.get(label_temp, label_temp)
                        color_selected, font_color_selected = theme_v
#                         graph_2.edge(src, tgt, label=label_to_sanskrit_form_iast.get(label_temp, label_temp), color="#74c476", fontcolor ="#6d000b")
                    
                    elif (label_temp == 'Oblique nominal' and "Case=Loc" in child_grammar):
                        value = "अधिकरणम्"
                        color_selected, font_color_selected = theme_v
#                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
                    elif (label_temp == 'Oblique nominal' and "Case=Dat" in child_grammar):
                        value = "सम्प्रदानम्"
                        color_selected, font_color_selected = theme_v
                    
                    elif (label_temp == 'Oblique nominal' and "Case=Abl" in child_grammar):
                        value = "अपादानम्"
                        color_selected, font_color_selected = theme_v
                    
                    elif ((label_temp == 'Oblique nominal') and ("Case=Acc" in child_grammar) and (has_child_with_label(tgt, 'case', children_dict))):
                        value = "कर्मप्रवचनीय_अन्वितः"
                        print("Sentence_ID:",group_id_list[i])
                        print("कर्मप्रवचनीय_अन्वितः")
                        print(parent_word1,"=>",child_word)
                        print(has_child_with_label(tgt, 'case', children_dict))
                        print(edges)
                        
#                         print(children_dict)
                        
                        color_selected, font_color_selected = theme_v
                    
                    elif (label_temp == 'Oblique nominal' and "Case=Acc" in child_grammar):
                        value = "कर्म"
#                         print("कर्म")
#                         print(edges)
#                         print(src,tgt)
                        color_selected, font_color_selected = theme_v
                        
                    
#                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
                    elif (label_temp == 'Oblique nominal' and "Case=Ins" in child_grammar and ("Ṇyat" in parent_grammar or "Gdv" in parent_grammar)):
                        value = "कर्ता" 
                        color_selected, font_color_selected = theme_v
#                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
                    elif (label_temp == 'Oblique nominal') and ("Case=Ins" in child_grammar):
                        value = "करणम्"
                        color_selected, font_color_selected = theme_v
#                         graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    
                    elif (label_temp == "Coordinating conjunction") and (child_word  == "vā"):
                        value = "अन्यतर_द्योतकः"
                        color_selected, font_color_selected = theme_v                    
                    
                    elif (label_temp == 'Nominal subject'):
                        if "Ṇyat" in parent_grammar or "Gdv" in parent_grammar:
                            value = "कर्म" 
                            color_selected, font_color_selected = theme_v
#                             graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                        else:
                            value = "कर्ता"
                            color_selected, font_color_selected = theme_v
#                             graph_2.edge(src, tgt, label= transliterate.process('Devanagari', 'IAST', value).capitalize(), color="#74c476", fontcolor ="#63006d")
                    else:
                        value = label_to_sanskrit_form_iast.get(label_temp, label_temp)
                        color_selected, font_color_selected = theme_g
#                         graph_2.edge(src, tgt, label=label_to_sanskrit_form_iast.get(label_temp, label_temp), color="#74c476", fontcolor ="#006d2c")
                    
#                     transformed_edges_set_1.append([src,tgt,transformed_label_set1])            
                elif label_temp in transform_1_labels:
                    value = label_to_sanskrit_form_iast.get(label_temp, label_temp)
                    color_selected, font_color_selected = theme_m
#                     graph_2.edge(src, tgt, label=label_to_sanskrit_form_iast.get(label_temp, label_temp), color="#74c476", fontcolor ="#6d000b")
                else:
                    value = label_to_sanskrit_form_iast.get(label_temp, label_temp)
                    color_selected, font_color_selected = theme_g
#                     graph_2.edge(src, tgt, label=label_to_sanskrit_form_iast.get(label_temp, label_temp), color="#74c476", fontcolor ="#006d2c")
                value_transliterated = transliterate.process('Devanagari', 'IAST', value).capitalize()
                transformed_edges_set_1.append((src, tgt, value_transliterated))
                graph_2.edge(src, tgt, label=value_transliterated, color=color_selected, fontcolor = font_color_selected)
            
            
            
            
            # Render and save the graph
            graph_1_filepath = "dependency_graph_1"
            graph_2_filepath = "dependency_graph_2"
            graph_1.render(graph_1_filepath, format="png", cleanup=True)
            graph_2.render(graph_2_filepath, format="png", cleanup=True)
            
            

            # Display the rendered graph
#             print("\033[1mMantra Text:\033[0m",text_combined_list[i])
#             print("\033[1mSentence ID:\033[0m", group_id_list[i])
# #             i = i+1
#             from IPython.display import Image, display
#             print("\033[1mGraph Style:\033[0m", "DCS Original")
#             display(Image(filename=graph_1_filepath + ".png")) 
#             print("\033[1mGraph Style:\033[0m", "Modified from DCS")
#             display(Image(filename=graph_2_filepath + ".png"))
#             print("First Transformeation Edges:")
#             for edge in transformed_edges_set_1:
#                 print(edge)
            
            """The Rule order is Rule 2, Rule 4, Rule 3, and finally Rule1
            The following rules have been applied:
            Rule 1a)
            if R1 is NOT one of the following -- kartaa, karma, lara.na, apaadaana, and sampradaana
            A <--- conjunct ---- B <----- R1 ----- C
            ====>    A <----- R1 ----- C  and   B <---- R1 ----- C

            Rule 1b)
            If R1 is one of the following -- kartaa, karma, lara.na, apaadaana, and sampradaana
            A <--- conjunct ---- B <----- R1 ----- C
            ====>    A <----- विशेषणम्  ----- B  and   B <---- R1 ----- C

            2.	For adjacent words:
            X <---- Conjunct ---- X 
            =====>  X <---- वीप्सा ----- X
            (Note both the words at the arrowhead and the arrow tail are the SAME)

            3.	A <----- विशेषणम्  ----- B <----- विशेषणम् ------ C
            =====> A <----- विशेषणम्  ----- C  and B <----- विशेषणम्  ----- C

            4.	For समिच्चित_द्योतकः, if parent <----- Conjuct----- grandparent
            Then add child to grandparent with link type समिच्चित_द्योतकः
            
                        
            5. A <--- समिच्चित_द्योतकः ---- B <--- conjunct ---- C <----- R1 ----- D 
            ====>    B <----- conjunct ----- D  and   C <---- R1 ----- D

            
            """
            print("Sentence_ID:",group_id_list[i])
            transformed_edges_set_1_a = transform_dependency_graph_rule_5(nodes, transformed_edges_set_1, output_prefix="dependency_graph")
            transformed_edges_set_2 = transform_dependency_graph_rule_2(nodes, transformed_edges_set_1_a, output_prefix="dependency_graph")
            transformed_edges_set_3 = transform_dependency_graph_rule_4(nodes, transformed_edges_set_2, output_prefix="dependency_graph")
            transformed_edges_set_4 = transform_dependency_graph_rule_3(nodes, transformed_edges_set_3, output_prefix="dependency_graph")
            transformed_edges_set_5 = transform_dependency_graph_rule_1(nodes, transformed_edges_set_4, output_prefix="dependency_graph", g = False)
            if (False):
                print("Transformation Edges 1:")
                for edge in transformed_edges_set_1:
                    print(edge)
                print("Transformation Edges 2:")
                for edge in transformed_edges_set_2:
                    print(edge)
                print("Transformation Edges 3:")
                for edge in transformed_edges_set_3:
                    print(edge)
                print("Transformation Edges: 4")
                for edge in transformed_edges_set_4:
                    print(edge)
                print("Transformation Edges: 5 ")
                for edge in transformed_edges_set_5:
                    print(edge)
            graph_reduction = Digraph(format='png', engine='dot')
    
            # Add nodes
            for node, label in nodes.items():
                graph_reduction.node(node, label=f"{label}",color="#cf3759",shape="box", style="rounded,filled", fillcolor="#FFEEEB", fontcolor ="#93003a")

            # Add original edges
            for src, tgt, label in transformed_edges_set_5:
                graph_reduction.edge(src, tgt, label=label, color="#f4777f", fontcolor ="#93003a")

            # Render and display
           
            reduction_path = f"dependency_graph_3"
            graph_reduction.render(reduction_path, format="png", cleanup=True)
#             print("\033[1mGraph Style:\033[0m", "Reduced from Modified Graph")
#             display(Image(filename=reduction_path + ".png"))
            
            
            heading = doc.add_heading(f"Sentence ID: {group_id_list[i]}", level=2)
            apply_font_style(heading, font_name='Calibri', size=14, bold=True)

            caption_v = doc.add_paragraph()
            run = caption_v.add_run(f"Mantra Text:\n{text_combined_list[i]} \n{transformed_edges_set_5}")
#             run.bold = True
            run.font.size = Pt(12)
            

            # Add dependency parse
#             parse_paragraph = doc.add_paragraph(f"Dependency Parsing from ByT5:\n{parse[num]}")
#             apply_font_style(parse_paragraph, font_name='Calibri', size=12)

            # Adding images
            image_width_limit = 6
            image_path_1 = graph_1_filepath + ".png"
        
            with PILImage.open(image_path_1) as img:
                width_px, height_px = img.size
                dpi = img.info.get('dpi', (96, 96))[0]  # Default to 96 DPI if not specified
                width_in = width_px / dpi
            
#             caption_v = doc.add_paragraph()
#             run = caption_v.add_run(f"Width pixel:{width_px} \n Width inch:{width_in}")
            # Add the image with resizing only if width > 4 inches
            if width_in > image_width_limit:
                doc.add_picture(image_path_1, width=Inches(image_width_limit))
            else:
                doc.add_picture(image_path_1, width=Inches(width_in))
#             doc.add_picture(image_path_1, width=Inches(4))
    
            caption_1 = doc.add_paragraph()
            run = caption_1.add_run(f"Figure {num * 3 + 1}: DCS Original Dependency Graph for Sentence ID {group_id_list[i]}")
            run.italic = True
            run.font.size = Pt(10)
            caption_1.paragraph_format.space_after = Pt(20)
            
            image_path_2 = graph_2_filepath + ".png"
#             doc.add_picture(image_path_2, width=Inches(4))
            with PILImage.open(image_path_2) as img:
                width_px, height_px = img.size
                dpi = img.info.get('dpi', (96, 96))[0]  # Default to 96 DPI if not specified
                width_in = width_px / dpi
#             caption_v = doc.add_paragraph()
#             run = caption_v.add_run(f"Width pixel:{width_px} \n Width inch:{width_in}")
            # Add the image with resizing only if width > 4 inches
            if width_in > image_width_limit:
                doc.add_picture(image_path_2, width=Inches(image_width_limit))
            else:
                doc.add_picture(image_path_2, width=Inches(width_in))
   
            caption_2 = doc.add_paragraph()
            run = caption_2.add_run(f"Figure {num * 3 + 2}: Sanskrit Tags Dependency Graph for Sentence ID {group_id_list[i]}")
            run.italic = True
            run.font.size = Pt(10)
            caption_2.paragraph_format.space_after = Pt(20)

            image_path_3 = reduction_path + ".png"
#             doc.add_picture(image_path_3, width=Inches(4))
            with PILImage.open(image_path_3) as img:
                width_px, height_px = img.size
                dpi = img.info.get('dpi', (96, 96))[0]  # Default to 96 DPI if not specified
                width_in = width_px / dpi
#             caption_v = doc.add_paragraph()
#             run = caption_v.add_run(f"Width pixel:{width_px} \n Width inch:{width_in}")
            # Add the image with resizing only if width > 4 inches
            if width_in > image_width_limit:
                doc.add_picture(image_path_3, width=Inches(image_width_limit))
            else:
                doc.add_picture(image_path_3, width=Inches(width_in))
            
            
            caption_3 = doc.add_paragraph()
            run = caption_3.add_run(f"Figure {num * 3 + 3}: Reduced Dependency Graph for Sentence ID {group_id_list[i]}")
            run.italic = True
            run.font.size = Pt(10)
            run.add_break(WD_BREAK.PAGE)
            
            i = i+1
            num = num + 3
            
#             break

doc.save("Reduced_RigVeda_Dependency_Graph_DCS_Ver_x7.docx")
print("Document successfully created!")

SyntaxError: invalid syntax (1078641783.py, line 307)

Sentence_ID: 576368
[('0_5', '0_1', 'nsubj'), ('0_1', '0_2', 'discourse'), ('0_5', '0_3', 'obl'), ('0_3', '0_4', 'case'), ('0_5', '0_6', 'advmod'), ('0_8', '0_7', 'obj'), ('0_5', '0_8', 'obl')]
Sentence_ID: 576369
[('0_3', '0_1', 'nsubj'), ('0_1', '0_2', 'appos')]
Sentence_ID: 576370
[('0_3', '0_1', 'nsubj'), ('0_3', '0_2', 'amod'), ('0_6', '0_4', 'advmod'), ('0_6', '0_5', 'obj'), ('0_6', '0_7', 'obl')]
Sentence_ID: 576371
[('0_4', '0_1', 'advmod'), ('0_4', '0_2', 'obl'), ('0_4', '0_3', 'obj')]
Sentence_ID: 576372
[('0_4', '0_1', 'vocative'), ('0_4', '0_2', 'obl'), ('0_4', '0_3', 'obj'), ('0_6', '0_5', 'nummod'), ('0_2', '0_6', 'conj'), ('0_6', '0_7', 'cc')]
Sentence_ID: 576373
0_4 -> Adhikaraṇam -> 0_2 -> Samuccita -> 0_6 Samuccita_dyotakaḥ -> 0_7
[('0_3', '0_1', 'nsubj'), ('0_3', '0_2', 'amod')]
Sentence_ID: 576374
[('0_2', '0_1', 'orphan'), ('0_2', '0_3', 'orphan'), ('0_6', '0_4', 'obj'), ('0_6', '0_5', 'obl'), ('0_2', '0_6', 'conj')]
Sentence_ID: 576375
[('0_1', '0_2', 'obl'), ('0_